# Transformer Full Training and PEFT Training Tradeoffs Analysis

In [1]:
from pathlib import Path
import pandas as pd
import json
from sklearn.metrics import confusion_matrix

base_path = Path.cwd().parent
bert_df = pd.read_csv(base_path/"data"/"transformer_models"/"bert_df.csv")
distilbert_df = pd.read_csv(base_path/"data"/"transformer_models"/"distilbert_df.csv")

full_train_df = pd.read_csv(base_path/"data"/"distilbert_df.csv")
peft_df = pd.read_csv(base_path/"data"/"peft_distilbert_df.csv")

In [2]:
cm_bert = confusion_matrix(bert_df['labels'], bert_df['predictions'], labels=[0, 1, 2, 3])
cm_distilbert = confusion_matrix(distilbert_df['labels'], distilbert_df['predictions'], labels=[0, 1, 2, 3])

print("Two Models Comparison\n")
print("BERT full fine-tune Confusion Matrix") 
print(cm_bert)

print("\nDistilBERT full fine-tune Confusion Matix")
print(cm_distilbert)

Two Models Comparison

BERT full fine-tune Confusion Matrix
[[1750   28   68   54]
 [  12 1875    9    4]
 [  35    7 1709  149]
 [  27    8   83 1782]]

DistilBERT full fine-tune Confusion Matix
[[1775   17   53   55]
 [   9 1871   13    7]
 [  45    6 1675  174]
 [  28    9  103 1760]]


In [3]:
cm_full_train = confusion_matrix(full_train_df['labels'], full_train_df['predictions'], labels=[0,1,2,3])

cm_peft = confusion_matrix(peft_df['labels'], peft_df['predictions'], labels=[0, 1, 2, 3])

print("PEFT Training vs Full Fine-Tuning Comparison\n")
print("DistilBert full fine-tune Confusion Matrix") 
print(cm_full_train)

print("\nPEFT Confusion Matix")
print(cm_peft)

PEFT Training vs Full Fine-Tuning Comparison

DistilBert full fine-tune Confusion Matrix
[[1784   22   51   43]
 [   6 1882    7    5]
 [  28    8 1735  129]
 [  22   11  109 1758]]

PEFT Confusion Matix
[[1773   22   58   47]
 [  15 1871    9    5]
 [  55    7 1679  159]
 [  37    7  100 1756]]


The predictive accuracy of each class are not equally distributed although both models' accuracy are similar. The poorest predictive performance was in class label 2 and the best performing was in class label 1. An important insight from the confusion matrix display above include: 
1. The class label 1 has the least false positive and false negative predictions.
2. The class labels 2 and 3 were the highest class where the labels were confused and classed as the other.

### Model metrics comparisons

In [4]:
def load_metrics(path): 
    with open(path) as p:
        return json.load(p)

bert_model_metrics = load_metrics(base_path/"results"/"transformer_models"/"bert"/"test_metrics.json")
bert_train_time = load_metrics(base_path/"results"/"transformer_models"/"bert"/"train_time.json")
distilbert_model_metrics = load_metrics(base_path/"results"/"transformer_models"/"distilbert"/"test_metrics.json")
distilbert_train_time = load_metrics(base_path/"results"/"transformer_models"/"distilbert"/"train_time.json") 


## updating contents of the dictionary 
del bert_model_metrics['cm'] 
del distilbert_model_metrics['cm']

bert_model_metrics["prediction_time"] = bert_model_metrics.pop("inference_time")
distilbert_model_metrics["prediction_time"] = distilbert_model_metrics.pop("inference_time")

bert_model_metrics["Training_time"] = bert_train_time["Training_time"]
distilbert_model_metrics["Training_time"] = distilbert_train_time["Training_time"]

bert_model_metrics["model"] = "BERT full fine-tune"
distilbert_model_metrics["model"] = "DistilBERT full fine-tune"

In [5]:
pd.DataFrame([bert_model_metrics, distilbert_model_metrics]).set_index("model")

,accuracy,precision,recall,f1,num_of_params,prediction_time,Training_time
model,,,,,,,
BERT full fine-tune,0.936316,0.936316,0.936316,0.936316,109485316,535.396,30810.603
DistilBERT full fine-tune,0.931711,0.931711,0.931711,0.931711,66956548,261.826,25714.772


#### Summary and Recommendation
Both Bert and distilBert model performances are similar for the AG-News dataset. The BERT model has more trainable parameters compared to the distilBERT model. But this is as expected because distilbert is a smaller, faster and distilled version of the BERT model.  Therefore the increase in training and prediction time as well as the slight increase in performance parameters for the BERT model is as expected for the model. 

It can be seen from the confusion matrix performances and test metrics that BERT performed more than the distilBERT by a very minimal margin.  But, this came at the cost of higher training parameters, train time and prediction time which overall require more compute resources.  The marginal improvement does not compensate for the extra cost incurred by the BERT.  Therefore, from a business objective, I will recommend the deployment and use of the distilBERT model over the BERT model. 

### PEFT vs Full fine-tuning performance comparisons

In [6]:
ft_test_metrics = load_metrics(base_path/'results'/'distilbert'/'test_metrics.json')
ft_train_time = load_metrics(base_path/'results'/'distilbert'/'train_time.json' )

peft_train_time = load_metrics(base_path/'results'/'peft'/'distilbert'/'peft_train_time.json')
peft_test_metrics = load_metrics(base_path/'results'/'peft'/'distilbert'/'peft_test_metrics.json')
peft_params = load_metrics(base_path/'results'/'peft'/'distilbert'/'peft_params.json')

del peft_test_metrics['cm']
del ft_test_metrics['cm']

peft_test_metrics["training_time"] = peft_train_time["Training_time"]
ft_test_metrics["training_time"] = ft_train_time["Training_time"]
peft_test_metrics["model_type"] = "PEFT_distilbert"
ft_test_metrics["model_type"] = "FT_distilbert"

model_metrics = pd.DataFrame([peft_test_metrics, ft_test_metrics]).set_index("model_type")
model_metrics

,accuracy,precision,recall,f1,inference_time,num_of_params,training_time
model_type,,,,,,,
PEFT_distilbert,0.931447,0.931447,0.931447,0.931447,22.884,67697672,6250.328
FT_distilbert,0.941974,0.941974,0.941974,0.941974,24.362,66956548,8520.363


### Number of trainable parameters for PEFT model 

In [7]:
peft_trainable_params = load_metrics(base_path/"results"/"peft"/"distilbert"/"peft_params.json")
pd.DataFrame(peft_trainable_params, index=["Peft_distilbert_Model"])

,trainable_params,all_params,trainable_percentage
Peft_distilbert_Model,741124,67697672,1.094756


## Conclusion 

The performance of the  full fine-tuning and the PEFT are comparable as expected because both training types used the same transformer model.  However, the compute memory and time are significantly different  between a fully trained transformer and the PEFT trained transformer with approximately 1% of the entire parameters trainable with PEFT model.  Therefore the 1% gain in performance for a fully trained transformer does not fully compensate for the cost in time, compute power and resources used in training the model.  Finally, the size of the PEFT trained model is approximately 4MB which is significantly smaller than the fully trained model which is 256MB in size. 